# Sanity checks: filtered U.S. panel vs. Fama-French market

Compares the value-weighted (VW) and equally weighted (EW) portfolios of the filtered Datastream panel
with the Fama-French market (Mkt-RF + RF), monthly and daily, and runs a set of universe diagnostics.

**Expectations for the VW portfolio (monthly):** correlation and R² close to one, beta close to one,
alpha statistically indistinguishable from zero, small tracking error. The pass/check flags use the
heuristic thresholds in `datastream.evaluation.THRESHOLDS`.

All functions live in `src/datastream/evaluation.py`; this notebook only configures and displays.
Outputs (tables and figures) are written to `OUTPUT_DIR`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from datastream import evaluation as ev
from datastream.ff_data import get_ff_factors, prefetch_ff_cache

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.width", 160)

# ── configuration ───────────────────────────────────────────────────────────────────────────────────
DATA_PATH = Path("/Users/ralfkellner/Datastream/PriceData/US/processed")
PENNY_PERCENTILE = 0.2
PANEL_FILE = DATA_PATH / f"US_data_panel_filtered_{PENNY_PERCENTILE}.feather"
DELISTING_RETURN = -0.35          # must match the value used in scripts/02_filter.py
OUTPUT_DIR = DATA_PATH / f"sanity_checks_{PENNY_PERCENTILE}"
ROLLING_MONTHS = 36
ROLLING_DAYS = 252
SUBPERIODS = [                    # (label, start, end) - adjust to your sample
    ("1993-1999", "1993-01", "1999-12"),
    ("2000-2009", "2000-01", "2009-12"),
    ("2010-2019", "2010-01", "2019-12"),
    ("2020-",     "2020-01", "2099-12"),
]

# colors: first two slots of the default categorical palette (dataviz skill); FF is always slot 1
C_FF, C_DS = "#2a78d6", "#eb6834"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load data

In [ ]:
df = pd.read_feather(PANEL_FILE)
df["Date"] = pd.to_datetime(df["Date"])
print(f"{len(df):,} stock-days, {df['Stock'].nunique():,} stocks, {df['Date'].min():%Y-%m-%d} to {df['Date'].max():%Y-%m-%d}")

prefetch_ff_cache(silent=True)     # refreshes stale cache entries without prompting
ff_m = get_ff_factors(3, "monthly")
ff_d = get_ff_factors(3, "daily")
print(f"FF monthly up to {ff_m.index.max()}, FF daily up to {ff_d.index.max():%Y-%m-%d}")

## 2. Monthly portfolios and comparison with the FF market

In [ ]:
m = ev.monthly_panel(df)
pf_m = ev.monthly_portfolios(m)

stats_vw, res_vw = ev.compare_to_market(pf_m["VW"], ff_m, periods_per_year=12, hac_lags=6)
stats_ew, res_ew = ev.compare_to_market(pf_m["EW"], ff_m, periods_per_year=12, hac_lags=6)
monthly_stats = pd.DataFrame({"VW": stats_vw, "EW": stats_ew})
monthly_stats

In [ ]:
T = ev.THRESHOLDS
checks = pd.DataFrame([
    ("Correlation VW vs Mkt",            stats_vw["corr"],                   f">= {T['monthly_corr']}",       ev.flag(stats_vw["corr"], T["monthly_corr"])),
    ("R² of VW on Mkt-RF",               stats_vw["r2"],                     f">= {T['monthly_r2']}",         ev.flag(stats_vw["r2"], T["monthly_r2"])),
    ("Beta",                             stats_vw["beta"],                   f"in {T['monthly_beta']}",       ev.flag(stats_vw["beta"], T["monthly_beta"])),
    ("|t(alpha)| (HAC, 6 lags)",         abs(stats_vw["alpha_t_hac"]),       f"< {T['monthly_alpha_abs_t']}", "ok" if abs(stats_vw["alpha_t_hac"]) < T["monthly_alpha_abs_t"] else "CHECK"),
    ("Tracking error p.a.",              stats_vw["tracking_error_annual"],  f"< {T['monthly_te_annual']}",   "ok" if stats_vw["tracking_error_annual"] < T["monthly_te_annual"] else "CHECK"),
], columns=["check", "value", "rule", "status"]).set_index("check")
checks

In [ ]:
print(res_vw.summary())

**EW portfolio on FF3.** The EW portfolio overweights small stocks, so a clearly positive SMB loading,
a beta below one and a lower R² than for VW are expected. A negative SMB loading would suggest that
small stocks are underrepresented or filtered away.

In [ ]:
res_ew3 = ev.regress_on_factors(pf_m["EW"], ff_m, ["Mkt-RF", "SMB", "HML"], hac_lags=6)
res_vw3 = ev.regress_on_factors(pf_m["VW"], ff_m, ["Mkt-RF", "SMB", "HML"], hac_lags=6)
ff3_table = pd.DataFrame({
    "VW coef": res_vw3.params, "VW t (HAC)": res_vw3.tvalues,
    "EW coef": res_ew3.params, "EW t (HAC)": res_ew3.tvalues,
})
ff3_table.loc["R²"] = [res_vw3.rsquared, np.nan, res_ew3.rsquared, np.nan]
ff3_table

## 3. Over time: cumulative returns, cumulative difference, rolling fit

Level differences in the cumulative plot are expected to grow slowly. What matters are **breaks**: a
sudden widening of the gap or a dip in rolling correlation points to a period with a data problem
(e.g. coverage changes, stale market caps, a single stock with a wrong MV).

In [ ]:
aligned = pd.concat([pf_m["VW"], (ff_m["Mkt-RF"] + ff_m["RF"]).rename("FF_Mkt")], axis=1, join="inner").dropna()
cum = (1 + aligned).cumprod()
roll = ev.rolling_fit(pf_m["VW"], ff_m, ROLLING_MONTHS)

fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
x = aligned.index.to_timestamp()
axes[0].plot(x, cum["FF_Mkt"], color=C_FF, lw=2, label="FF market")
axes[0].plot(x, cum["VW"], color=C_DS, lw=2, label="Datastream VW")
axes[0].set_yscale("log"); axes[0].set_title("Cumulative return (log scale)"); axes[0].legend(frameon=False)
axes[1].plot(x, np.log(cum["VW"] / cum["FF_Mkt"]), color=C_DS, lw=2)
axes[1].axhline(0, color="grey", lw=1); axes[1].set_title("log(cumulative VW / cumulative FF market)")
axes[2].plot(roll.index.to_timestamp(), roll["corr"], color=C_DS, lw=2, label=f"{ROLLING_MONTHS}m correlation")
axes[2].plot(roll.index.to_timestamp(), roll["beta"], color=C_FF, lw=2, label=f"{ROLLING_MONTHS}m beta")
axes[2].axhline(1, color="grey", lw=1); axes[2].set_title("Rolling fit"); axes[2].legend(frameon=False)
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.3)
fig.tight_layout(); fig.savefig(OUTPUT_DIR / "monthly_vw_vs_ff.png", dpi=150)
plt.show()

In [ ]:
sub = ev.subperiod_table(pf_m["VW"], ff_m, SUBPERIODS, periods_per_year=12, hac_lags=6)
sub[["n_obs", "corr", "beta", "r2", "alpha_annual", "alpha_t_hac", "tracking_error_annual"]]

**Largest monthly deviations.** Inspect the stocks driving these months (see section 5 for weights).

In [ ]:
dev_m = ev.largest_deviations(pf_m["VW"], ff_m, n=10)
dev_m

## 4. Daily comparison

Event studies use daily data, so the daily VW portfolio should track the FF daily market as well.
Correlation is typically a bit lower than monthly (closing-price timing, holidays). Days with large
deviations are good candidates for a look at the raw data.

In [ ]:
pf_d = ev.daily_portfolios(df)
stats_vw_d, res_vw_d = ev.compare_to_market(pf_d["VW"], ff_d, periods_per_year=252, hac_lags=5)
stats_ew_d, _ = ev.compare_to_market(pf_d["EW"], ff_d, periods_per_year=252, hac_lags=5)
daily_stats = pd.DataFrame({"VW": stats_vw_d, "EW": stats_ew_d})
print("Daily correlation check:", ev.flag(stats_vw_d["corr"], T["daily_corr"]))
daily_stats

In [ ]:
missing_in_ds = ff_d.index.difference(pf_d.index)
missing_in_ds = missing_in_ds[(missing_in_ds >= pf_d.index.min()) & (missing_in_ds <= pf_d.index.max())]
extra_in_ds = pf_d.index.difference(ff_d.index)
extra_in_ds = extra_in_ds[extra_in_ds <= ff_d.index.max()]
print(f"FF trading days missing in the panel: {len(missing_in_ds)}; panel days that are not FF trading days: {len(extra_in_ds)}")
print("Examples of panel days not in FF (holidays not removed by filter 16?):", list(extra_in_ds[:10].strftime("%Y-%m-%d")))

In [ ]:
dev_d = ev.largest_deviations(pf_d["VW"], ff_d, n=15)
dev_d

In [ ]:
roll_d = ev.rolling_fit(pf_d["VW"], ff_d, ROLLING_DAYS)
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(roll_d.index, roll_d["corr"], color=C_DS, lw=1.5, label=f"{ROLLING_DAYS}d correlation")
ax.plot(roll_d.index, roll_d["beta"], color=C_FF, lw=1.5, label=f"{ROLLING_DAYS}d beta")
ax.axhline(1, color="grey", lw=1); ax.legend(frameon=False); ax.set_title("Daily VW vs FF market: rolling fit")
ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.3)
fig.tight_layout(); fig.savefig(OUTPUT_DIR / "daily_rolling_fit.png", dpi=150)
plt.show()

## 5. Universe diagnostics

* **Number of stocks:** should evolve smoothly (U.S. listings peak in the late 1990s and decline
  afterwards). Jumps point to download or filter problems.
* **Max single-stock weight:** in the U.S. market the largest stock rarely exceeds ~7-8%. A higher weight
  usually means a wrong market cap (units, stale MV, duplicate share classes).
* **Implied share changes:** stock-months where market cap growth and return disagree by more than
  a factor of 3 - candidates for MV errors, sorted by size (large stocks matter most for VW).

In [ ]:
weights = ev.monthly_weights(m)
uni = ev.universe_over_time(m, weights)

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
xu = uni.index.to_timestamp()
axes[0].plot(xu, uni["n_stocks"], color=C_DS, lw=2); axes[0].set_title("Stocks in the VW portfolio")
axes[1].plot(xu, uni["total_mcap_lag"] / 1e6, color=C_DS, lw=2); axes[1].set_title("Total market cap (lagged, trillions if MV is in millions)")
axes[2].plot(xu, uni["max_weight"], color=C_DS, lw=2, label="largest stock")
axes[2].plot(xu, uni["top10_weight"], color=C_FF, lw=2, label="top 10 stocks")
axes[2].axhline(T["max_weight"], color="grey", lw=1, ls="--"); axes[2].set_title("Portfolio concentration"); axes[2].legend(frameon=False)
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", alpha=0.3)
fig.tight_layout(); fig.savefig(OUTPUT_DIR / "universe_over_time.png", dpi=150)
plt.show()

In [ ]:
print(f"Months with a single-stock weight above {T['max_weight']:.0%}: {(uni['max_weight'] > T['max_weight']).sum()}")
uni.sort_values("max_weight", ascending=False).head(10)

In [ ]:
mv_flags = ev.mcap_consistency(m, threshold=3.0)
print(f"{len(mv_flags):,} stock-months with implied share change > 3x or < 1/3x")
mv_flags.head(20)

## 6. Return profile

Landis & Skouras (2021) report about 30% zero daily returns in their filtered U.S. sample (23% in CRSP)
and per-stock daily volatilities between 0.11% and 40%. Much higher zero shares suggest stale data or
low-precision downloads.

In [ ]:
profile = ev.daily_return_profile(df)
profile

In [ ]:
ev.stock_volatility_profile(df)

## 7. Impact of the delisting return

Compares portfolios with and without the delisting adjustment (-35% on the last observation of each
delisted stock). The EW portfolio should react more than VW, because delisted stocks are mostly small.

In [ ]:
df_nodl = ev.undo_delisting_adjustment(df, DELISTING_RETURN)
pf_m_nodl = ev.monthly_portfolios(ev.monthly_panel(df_nodl))
delisting_impact = pd.DataFrame({
    "with delisting return": pf_m.mean() * 12,
    "without delisting return": pf_m_nodl.mean() * 12,
})
delisting_impact["difference p.a."] = delisting_impact.iloc[:, 0] - delisting_impact.iloc[:, 1]
stats_vw_nodl, _ = ev.compare_to_market(pf_m_nodl["VW"], ff_m, 12, 6)
print(f"VW vs FF market without delisting return: corr {stats_vw_nodl['corr']:.4f}, beta {stats_vw_nodl['beta']:.4f}, "
      f"alpha p.a. {stats_vw_nodl['alpha_annual']:.4f}")
delisting_impact

## 8. Save results

In [ ]:
checks.to_csv(OUTPUT_DIR / "checks_summary.csv")
monthly_stats.to_csv(OUTPUT_DIR / "monthly_stats.csv")
daily_stats.to_csv(OUTPUT_DIR / "daily_stats.csv")
ff3_table.to_csv(OUTPUT_DIR / "ff3_regressions.csv")
sub.to_csv(OUTPUT_DIR / "subperiods_vw.csv")
dev_m.to_csv(OUTPUT_DIR / "largest_deviations_monthly.csv")
dev_d.to_csv(OUTPUT_DIR / "largest_deviations_daily.csv")
uni.to_csv(OUTPUT_DIR / "universe_over_time.csv")
mv_flags.to_csv(OUTPUT_DIR / "mcap_consistency_flags.csv", index=False)
profile.to_csv(OUTPUT_DIR / "daily_return_profile.csv")
delisting_impact.to_csv(OUTPUT_DIR / "delisting_impact.csv")
pf_m.to_csv(OUTPUT_DIR / "portfolios_monthly.csv")
pf_d.to_csv(OUTPUT_DIR / "portfolios_daily.csv")
print(f"Saved to {OUTPUT_DIR}")